# Stage 2 — Data Cleaning & Preprocessing
**Project:** LendingClub Loan Default Prediction  
**Methodology:** CRISP-DM  
**Notebook:** `02_cleaning_preprocessing.ipynb`

---

## Purpose
Transform raw loan data from SQLite into a clean, split, model-ready parquet file.  
This notebook is the **single source of truth** for all data decisions — every downstream notebook loads from the artifacts produced here.

---

## Stage 2 Concept Summary

### Why we filter years before splitting
Loans issued in 2017–2018 haven't had enough time to reach their true terminal status (Charged Off / Fully Paid). Many that will eventually default are still labelled "Current" — a **label maturity problem**. Training or testing on these rows corrupts both the signal and the evaluation. Fix: keep only `issue_year <= 2016`.

### The golden rule: split before fitting any statistic
Imputation medians, encoding maps, and scalers must be **fit on train only**, then applied to both train and test. Fitting on the full dataset lets test-set information leak into training — your evaluation metrics will be optimistic and your model will underperform on real new data.

### Dtype optimisation matters on M1
Pandas defaults to `float64`/`int64` (8 bytes each). Most columns in this dataset fit in `float32` or `int16`. A downcast pass after loading cuts RAM use by ~50%, which is critical on a 8GB MacBook Air.

### Target encoding for high-cardinality categoricals
`addr_state` has 51 levels. One-hot would add 50 sparse binary columns. Target encoding replaces each state with its mean default rate from the **train set only** — 1 column, real signal, zero leakage if done correctly. Unseen states in test are filled with the global train mean.

---

## Order of Operations
```
1. Load with column filter     ← drop dead columns at SQL time, never touch them
2. Structural cleaning         ← string stripping, ordinal encoding, dtype downcast
3. Filter to issue_year ≤ 2016 ← remove immature loan vintages
4. Build target variable       ← loan_status → binary 0/1
5. Train/test split            ← stratified 80/20, save indices
6. Imputation                  ← fit medians on train, apply to both
7. Encoding                    ← one-hot + target encode, fit on train only
8. Save artifacts              ← parquet + split indices JSON
```

---

## Artifacts Produced
| File | Contents |
|---|---|
| `data/02_cleaned.parquet` | Final cleaned + encoded dataframe (features + target) |
| `data/02_split_indices.json` | Train/test row indices for reproducibility |
| `data/leakage_columns.json` | Updated with settlement_* columns (addendum from Stage 1) |

---
## 0. Imports & Config
Load all libraries upfront. Setting display options here keeps all subsequent outputs clean and consistent.

In [1]:
import sqlite3
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ────────────────────────────────────────────────────────────────────
DB_PATH   = 'lending_club.db'
DATA_DIR  = Path('data')
DATA_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42

print('Libraries loaded.')
print(f'Data directory: {DATA_DIR.resolve()}')

Libraries loaded.
Data directory: /Users/seihavat/data


---
## 1. Build the Column Exclusion List

Before touching the database, we compile every column we've decided to exclude.  
This list comes from three sources identified in Stages 0 and 1:
- **Leakage columns** — post-origination fields from `leakage_columns.json`
- **Settlement addendum** — 5 post-default fields missed in Stage 0 audit (flagged in Stage 1)
- **Structural drops** — identifiers, free-text, and >50% missing columns

Loading only safe columns from SQL means these bytes never enter RAM.

In [2]:
# 1. Load the dictionary
with open(DATA_DIR / 'leakage_columns.json', 'r') as f:
    leakage_dict = json.load(f)

# 2. Flatten the dictionary into a set immediately
# This is cleaner than creating an empty list and using extend()
leakage_cols = set()
for group_name, column_list in leakage_dict.items():
    leakage_cols.update(column_list)

# 3. Add the Stage 1 fields
settlement_cols = {
    'settlement_term',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'debt_settlement_flag_date',
}
leakage_cols.update(settlement_cols)

# ── Identifiers and free-text fields ─────────────────────────────────────────
# These carry no generalizable signal: IDs are unique per row,
# free-text requires NLP which is out of scope for this project.
identifier_cols = {
    'member_id', 'url', 'desc', 'title', 'zip_code', 'emp_title'
}

# ── >50% missing columns identified in Stage 1 EDA ───────────────────────────
# These are structurally sparse — not randomly missing, but absent by design
# (e.g. hardship_* only exists for the 0.5% of loans that entered hardship).
# Imputing them would fabricate data for 99.5% of rows.
high_missing_cols = {
    # Hardship group (99.5% missing)
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date',
    'hardship_end_date', 'payment_plan_start_date', 'hardship_length',
    'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    # Other structurally sparse columns (>50% missing from Stage 1)
    'mths_since_recent_bc_dlq', 'mths_since_last_major_derog',
    'mths_since_recent_revol_delinq', 'annual_inc_joint', 'dti_joint',
    'verification_status_joint', 'il_util', 'mths_since_rcnt_il',
    'open_il_6m', 'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
    'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
    'open_acc_6m', 'total_bal_il', 'sec_app_fico_range_low',
    'sec_app_fico_range_high', 'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc',
    'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog', 'revol_bal_joint','settlement_percentage','open_act_il'
}

# ── Redundant columns ─────────────────────────────────────────────────────────
# grade is fully encoded by sub_grade (A1-G5) — keeping both is redundant.
# funded_amnt and loan_amnt are near-identical (r≈0.99 expected) — drop one.
# funded_amnt_inv is a funding-side field, not borrower-side.
redundant_cols = {'grade', 'funded_amnt_inv'}

# ── Combine all exclusions ────────────────────────────────────────────────────
all_exclude = leakage_cols | identifier_cols | high_missing_cols | redundant_cols

# ── Update leakage_columns.json with settlement addendum ─────────────────────
with open(DATA_DIR / 'leakage_columns.json', 'w') as f:
    json.dump(sorted(list(leakage_cols)), f, indent=2)

print(f'Leakage columns:    {len(leakage_cols)}')
print(f'Identifier columns: {len(identifier_cols)}')
print(f'High-missing cols:  {len(high_missing_cols)}')
print(f'Redundant cols:     {len(redundant_cols)}')
print(f'Total to exclude:   {len(all_exclude)}')

Leakage columns:    31
Identifier columns: 6
High-missing cols:  50
Redundant cols:     2
Total to exclude:   89


---
## 2. Load Data from SQLite (Column-Filtered)

We query only the columns we need. SQLite returns them as a dataframe — the excluded columns never enter RAM.  
We also load `loan_status` and `issue_d` here since we need them to build the target and filter years, even though they won't be model features.

In [3]:
# ── Inspect all available columns in the database ────────────────────────────
conn = sqlite3.connect(DB_PATH)

all_db_cols = pd.read_sql(
    "PRAGMA table_info(raw_loads)", conn
)['name'].tolist()

# Determine which columns to load
# Always keep: loan_status (target), issue_d (year filter)
always_keep = {'loan_status', 'issue_d'}
cols_to_load = [c for c in all_db_cols if (c not in all_exclude) or (c in always_keep)]
# Deduplicate while preserving order
cols_to_load = list(dict.fromkeys(cols_to_load))

print(f'Total columns in DB:  {len(all_db_cols)}')
print(f'Columns excluded:     {len(all_exclude)}')
print(f'Columns to load:      {len(cols_to_load)}')

Total columns in DB:  151
Columns excluded:     89
Columns to load:      66


In [4]:
# ── Load filtered columns from SQLite ────────────────────────────────────────
col_str = ', '.join([f'"{c}"' for c in cols_to_load])
query   = f'SELECT {col_str} FROM raw_loads'

df = pd.read_sql(query, conn)
conn.close()

print(f'Loaded shape:  {df.shape}')
print(f'Memory usage:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Loaded shape:  (2260701, 66)
Memory usage:  2766.0 MB


### Interpretation
Compare the loaded column count against the expected ~110 safe columns.  
If memory is above ~1.5 GB at this point, check whether any high-missing columns slipped through the exclusion list.

---
## 3. Filter to Mature Loan Vintages (issue_year ≤ 2016)

Loans issued in 2017 and 2018 haven't had enough time to charge off — their labels are immature.  
We extract the issue year from `issue_d`, filter, then confirm the row count matches the ~1.26M expected from Stage 1.

In [5]:
# ── Parse issue year from 'issue_d' (format: 'Jan-2015') ─────────────────────
df['issue_year'] = pd.to_datetime(df['issue_d'], format='%b-%Y').dt.year

# ── Check distribution before filtering ──────────────────────────────────────
print('Row counts by issue year (pre-filter):')
print(df['issue_year'].value_counts().sort_index())

Row counts by issue year (pre-filter):
issue_year
2007.0000       603
2008.0000      2393
2009.0000      5281
2010.0000     12537
2011.0000     21721
2012.0000     53367
2013.0000    134814
2014.0000    235629
2015.0000    421095
2016.0000    434407
2017.0000    443579
2018.0000    495242
Name: count, dtype: int64


In [6]:
# ── Apply year filter ─────────────────────────────────────────────────────────
rows_before = len(df)
df = df[df['issue_year'] <= 2016].copy()
rows_after  = len(df)

# Drop issue_d and issue_year — no longer needed as model features
df.drop(columns=['issue_d', 'issue_year'], inplace=True)

print(f'Rows before filter: {rows_before:,}')
print(f'Rows after filter:  {rows_after:,}')
print(f'Rows dropped:       {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.1f}%)')

Rows before filter: 2,260,701
Rows after filter:  1,321,847
Rows dropped:       938,854 (41.5%)


In [7]:
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

Memory usage: 1553.90 MB


### Interpretation
Expected: ~1.32M rows after filtering. If significantly lower, check whether `issue_d` parsing failed for some years (would silently drop rows with NaT).  
If significantly higher, confirm that 2017 and 2018 are gone from the distribution — a recheck with `df['issue_year'].value_counts()` after the drop is worth running if in doubt.

---
## 4. Build Binary Target Variable

Map `loan_status` to a binary label using the definition established in Stage 0.  
Default = 1 (Charged Off, Default, Late 31-120, Does not meet credit policy: Charged Off).  
All other statuses = 0.  
After mapping, `loan_status` is dropped — it must never enter the feature set.

In [8]:
# ── Define default statuses from Stage 0 ─────────────────────────────────────
DEFAULT_STATUSES = {
    'Charged Off',
    'Default',
    'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off',
}

# ── Map to binary target ──────────────────────────────────────────────────────
df['target'] = df['loan_status'].isin(DEFAULT_STATUSES).astype('int8')
df.drop(columns=['loan_status'], inplace=True)

# ── Validate class distribution ───────────────────────────────────────────────
dist = df['target'].value_counts()
rate = df['target'].mean()

print('Target distribution:')
print(dist)
print(f'\nDefault rate:    {rate:.4f} ({rate*100:.2f}%)')
print(f'Imbalance ratio: {dist[0]/dist[1]:.2f}:1')

Target distribution:
target
0    1094288
1     227559
Name: count, dtype: int64

Default rate:    0.1722 (17.22%)
Imbalance ratio: 4.81:1


### Interpretation
Expected default rate: ~17% (after filter while Stage 1 finding of 12.9%).  
If the rate has shifted significantly, re-examine the year filter — immature 2017/2018 loans were artificially suppressing defaults, so filtering them out may nudge the rate slightly upward. A small increase is expected and correct.

---
## 5. Structural Cleaning

These transformations fix format issues that don't require computing statistics — safe to do before the train/test split.  
Three operations in this section:
- Strip and encode `term` (string → binary)
- Ordinal encode `sub_grade` (A1–G5 → 1–35)
- Ordinal encode `emp_length` (categorical years → ordered integer)

In [9]:
# ── term: ' 36 months' / ' 60 months' → 0 / 1 ───────────────────────────────
# Strip whitespace and map to binary. 60-month loans have ~1.6x higher
# default rate (Stage 1: 17.58% vs 10.96%) — this is a meaningful signal.
print('term value counts (raw):')
print(df['term'].value_counts())

df['term'] = df['term'].str.strip().map({'36 months': 0, '60 months': 1}).astype('int8')

print('\nterm after encoding:')
print(df['term'].value_counts())

term value counts (raw):
term
36 months    944664
60 months    377183
Name: count, dtype: int64

term after encoding:
term
0    944664
1    377183
Name: count, dtype: int64


In [10]:
# ── sub_grade: A1–G5 → ordinal 1–35 ─────────────────────────────────────────
# sub_grade captures LC's risk assessment at application time.
# A1 = lowest risk, G5 = highest risk. Monotonic relationship with default.
# Encoding as an ordered integer preserves this monotonic structure for the model.

grades  = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
numbers = ['1', '2', '3', '4', '5']
subgrade_map = {
    f'{g}{n}': (gi * 5 + int(n))
    for gi, g in enumerate(grades)
    for n in numbers
}
# A1 → 1, A2 → 2, ..., A5 → 5, B1 → 6, ..., G5 → 35

df['sub_grade'] = df['sub_grade'].map(subgrade_map).astype('int8')

print('sub_grade encoding sample:')
print({k: v for k, v in list(subgrade_map.items())[::7]})  # sample every 7th
print(f'\nsub_grade range: {df["sub_grade"].min()} – {df["sub_grade"].max()}')
print(f'Nulls: {df["sub_grade"].isna().sum()}')

sub_grade encoding sample:
{'A1': 1, 'B3': 8, 'C5': 15, 'E2': 22, 'F4': 29}

sub_grade range: 1 – 35
Nulls: 0


In [11]:
# ── emp_length: string categories → ordered integer ──────────────────────────
# Employment length is an application-time feature. Longer employment
# generally correlates with income stability — modest but real signal.
# 'n/a' (not employed / self-employed / not stated) gets its own bin (0)
# rather than being treated as missing — it carries information.

emp_length_map = {
    'n/a':      0,
    '< 1 year': 1,
    '1 year':   2,
    '2 years':  3,
    '3 years':  4,
    '4 years':  5,
    '5 years':  6,
    '6 years':  7,
    '7 years':  8,
    '8 years':  9,
    '9 years':  10,
    '10+ years': 11,
}

print('emp_length raw value counts:')
print(df['emp_length'].value_counts(dropna=False))

df['emp_length'] = df['emp_length'].map(emp_length_map)
# Any genuine NaN (not 'n/a' string) will be imputed in Section 7.

print(f'\nNulls after mapping: {df["emp_length"].isna().sum()}')

emp_length raw value counts:
emp_length
10+ years    441566
2 years      118474
3 years      104762
< 1 year     102526
1 year        86251
5 years       82184
4 years       78350
None          73049
8 years       62731
6 years       61608
7 years       58802
9 years       51544
Name: count, dtype: int64

Nulls after mapping: 73049


### Interpretation
All three encodings preserve the ordinal rank of the original categories.  
- `term`: 0 = 36 months (safer), 1 = 60 months (riskier) — binary, no ambiguity  
- `sub_grade`: 1–35, monotonically increasing risk — the model can treat this as near-continuous  
- `emp_length`: 0–11, with 0 = unknown/unemployed — any remaining nulls after this mapping are genuine gaps, imputed next

---
## 6. Dtype Optimisation

Downcast numeric columns to the smallest dtype that fits the data range.  
This is a memory operation only — no values change, just the storage format.  
Target: reduce total dataframe RAM by ~50%.

In [12]:
# ── Record memory before ──────────────────────────────────────────────────────
mem_before = df.memory_usage(deep=True).sum() / 1e6

# ── Downcast integers ─────────────────────────────────────────────────────────
int_cols = df.select_dtypes(include=['int64', 'int32']).columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')

# ── Downcast floats ───────────────────────────────────────────────────────────
# float32 gives 7 significant decimal digits — more than enough for
# financial ratios like dti (0.00–999.99) or revol_util (0.0–100.0)
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast='float')

# ── Encode low-cardinality string columns as category ─────────────────────────
# We'll handle these properly via encoders below; for now just track them.
obj_cols = df.select_dtypes(include='object').columns.tolist()

mem_after = df.memory_usage(deep=True).sum() / 1e6

print(f'Memory before downcast: {mem_before:.1f} MB')
print(f'Memory after downcast:  {mem_after:.1f} MB')
print(f'Reduction:              {(mem_before - mem_after)/mem_before*100:.1f}%')
print(f'\nRemaining object columns ({len(obj_cols)}): {obj_cols}')

Memory before downcast: 1272.6 MB
Memory after downcast:  1003.0 MB
Reduction:              21.2%

Remaining object columns (10): ['id', 'home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'application_type', 'disbursement_method']


### Interpretation
A 22.50 % memory reduction is typical for this dataset.  
The remaining object columns are the categorical features that need proper encoding — they can't be safely downcast until we decide their encoding strategy.  
If memory is still above 1 GB, check whether any float64 columns resisted downcasting (usually due to all-null columns — those should have been dropped already).

---
## 7. Train / Test Split

Stratified 80/20 split on the target variable.  
**Stratified** means both train and test will have the same ~12.9% default rate — without this, random chance could give test set a very different class ratio, making evaluation unreliable.  
We save the indices to disk so every downstream notebook uses the exact same split — reproducibility is non-negotiable.

In [13]:
# ── Separate features and target ──────────────────────────────────────────────
X = df.drop(columns=['target'])
y = df['target']

# ── Stratified split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f'Train rows: {len(X_train):,}  |  Default rate: {y_train.mean():.4f}')
print(f'Test rows:  {len(X_test):,}   |  Default rate: {y_test.mean():.4f}')

Train rows: 1,057,477  |  Default rate: 0.1722
Test rows:  264,370   |  Default rate: 0.1722


In [14]:
# ── Save split indices for reproducibility ────────────────────────────────────
# Every downstream notebook must load from this file rather than re-splitting.
# Re-splitting with a different seed would produce data leakage between stages.
split_indices = {
    'train_indices': X_train.index.tolist(),
    'test_indices':  X_test.index.tolist(),
    'random_seed':   RANDOM_SEED,
    'test_size':     0.2,
    'stratified':    True
}

with open(DATA_DIR / '02_split_indices.json', 'w') as f:
    json.dump(split_indices, f)

print('Split indices saved to data/02_split_indices.json')

Split indices saved to data/02_split_indices.json


### Interpretation
Both train and test default rates should match within 0.1% of each other — that's the stratification working correctly.  
If they differ by more, check whether `stratify=y` was applied. A mismatch here would invalidate every recall metric in Stage 5.

---
## 8. Missing Value Audit & Imputation

Three imputation strategies, applied by column group:
- **Median imputation** — continuous skewed features (income, utilisation, DTI). Median is robust to outliers; mean is not.
- **Sentinel imputation (999)** — `mths_since_last_delinq` and similar: missingness means *no history*, which is genuinely different from having a delinquency 12 months ago. A sentinel preserves that meaning.
- **Mode imputation** — low-cardinality categoricals with <5% missing.

**All statistics are computed on train only, then applied to test.**

In [15]:
# ── Audit missing values in train set ────────────────────────────────────────
missing = (
    X_train.isna().sum()
    .pipe(lambda s: s[s > 0])
    .sort_values(ascending=False)
    .to_frame('n_missing')
)
missing['pct_missing'] = (missing['n_missing'] / len(X_train) * 100).round(2)
missing['dtype']       = X_train[missing.index].dtypes.values

print(f'Columns with missing values: {len(missing)}')
print(missing.to_string())

Columns with missing values: 45
                            n_missing  pct_missing    dtype
num_tl_120dpd_2m                95324       9.0100  float32
mo_sin_old_il_acct              86114       8.1400  float32
emp_length                      58324       5.5200  float32
pct_tl_nvr_dlq                  56402       5.3300  float32
avg_cur_bal                     56284       5.3200  float32
num_rev_accts                   56275       5.3200  float32
tot_hi_cred_lim                 56274       5.3200  float32
num_rev_tl_bal_gt_0             56274       5.3200  float32
num_op_rev_tl                   56274       5.3200  float32
num_il_tl                       56274       5.3200  float32
num_bc_tl                       56274       5.3200  float32
num_actv_rev_tl                 56274       5.3200  float32
num_actv_bc_tl                  56274       5.3200  float32
num_accts_ever_120_pd           56274       5.3200  float32
mo_sin_rcnt_tl                  56274       5.3200  float32
mo_sin_o

In [16]:
# ── Sentinel imputation: mths_since_* columns ────────────────────────────────
# NaN here means "never happened" — encode as 999 so the model can learn
# that no-history is a distinct and informative state.

sentinel_cols = [
    'mths_since_last_delinq',
    'mths_since_last_record',
    'mths_since_recent_inq',
    'mths_since_recent_bc',
    'mths_since_last_major_derog',
]
# Filter to only columns that actually exist in the loaded data
sentinel_cols = [c for c in sentinel_cols if c in X_train.columns]

for col in sentinel_cols:
    X_train[col] = X_train[col].fillna(999)
    X_test[col]  = X_test[col].fillna(999)

print(f'Sentinel imputed ({len(sentinel_cols)} cols): {sentinel_cols}')

Sentinel imputed (1 cols): ['mths_since_recent_bc']


In [17]:
# ── Median imputation: continuous numeric columns ─────────────────────────────
# Fit medians on train only. These columns have <5% missing in most cases —
# median is safe and avoids distortion from extreme income outliers.

# Identify numeric columns still with missing values (after sentinel pass)
num_missing = [
    c for c in X_train.select_dtypes(include='number').columns
    if X_train[c].isna().sum() > 0
    and c not in sentinel_cols
]

# Compute medians on train
medians = X_train[num_missing].median()

# Apply to both train and test
X_train[num_missing] = X_train[num_missing].fillna(medians)
X_test[num_missing]  = X_test[num_missing].fillna(medians)

print(f'Median imputed ({len(num_missing)} cols):')
for col in num_missing:
    print(f'  {col}: median = {medians[col]:.4f}')

Median imputed (43 cols):
  emp_length: median = 7.0000
  annual_inc: median = 65000.0000
  dti: median = 17.8400
  inq_last_6mths: median = 0.0000
  open_acc: median = 11.0000
  pub_rec: median = 0.0000
  revol_util: median = 54.2000
  total_acc: median = 23.0000
  collections_12_mths_ex_med: median = 0.0000
  acc_now_delinq: median = 0.0000
  tot_coll_amt: median = 0.0000
  tot_cur_bal: median = 80804.0000
  total_rev_hi_lim: median = 24100.0000
  acc_open_past_24mths: median = 4.0000
  avg_cur_bal: median = 7412.0000
  bc_open_to_buy: median = 4363.0000
  bc_util: median = 65.8000
  chargeoff_within_12_mths: median = 0.0000
  delinq_amnt: median = 0.0000
  mo_sin_old_il_acct: median = 130.0000
  mo_sin_old_rev_tl_op: median = 167.0000
  mo_sin_rcnt_rev_tl_op: median = 8.0000
  mo_sin_rcnt_tl: median = 6.0000
  mort_acc: median = 1.0000
  num_accts_ever_120_pd: median = 0.0000
  num_actv_bc_tl: median = 3.0000
  num_actv_rev_tl: median = 5.0000
  num_bc_sats: median = 4.0000
  num_bc

In [18]:
# ── Mode imputation: categorical columns with missing values ─────────────────
# For low-cardinality categoricals (e.g. home_ownership, verification_status)
# the mode is the safest default. Fit on train only.

obj_missing = [
    c for c in X_train.select_dtypes(include='object').columns
    if X_train[c].isna().sum() > 0
]

modes = X_train[obj_missing].mode().iloc[0]

X_train[obj_missing] = X_train[obj_missing].fillna(modes)
X_test[obj_missing]  = X_test[obj_missing].fillna(modes)

print(f'Mode imputed ({len(obj_missing)} cols):')
for col in obj_missing:
    print(f'  {col}: mode = "{modes[col]}"')

Mode imputed (1 cols):
  earliest_cr_line: mode = "Aug-2001"


In [19]:
# ── Verify: no missing values remain ─────────────────────────────────────────
train_nulls = X_train.isna().sum().sum()
test_nulls  = X_test.isna().sum().sum()

print(f'Remaining nulls — Train: {train_nulls}  |  Test: {test_nulls}')

if train_nulls > 0 or test_nulls > 0:
    print('\n⚠️  Columns still with nulls:')
    remaining = X_train.columns[X_train.isna().any()].tolist()
    print(remaining)

Remaining nulls — Train: 0  |  Test: 0


### Interpretation
Both train and test should show 0 remaining nulls after this section.  
If any columns resist imputation, they're likely new object columns that weren't caught by the mode pass — inspect their value counts and decide: drop or encode first, then impute.

---
## 9. Categorical Encoding

Two encoding strategies for the remaining object columns:

**One-hot encoding** — for low-cardinality nominals (`home_ownership`, `verification_status`, `purpose`, `initial_list_status`, `application_type`, `disbursement_method`).  
These have no natural order, so ordinal encoding would imply a false ranking.

**Target encoding** — for `addr_state` (51 levels). Replace each state with its mean default rate from the train set.  
One column, real signal, no false ordinality. Unseen states in test → global train mean.

In [20]:
# ── Target encode addr_state ──────────────────────────────────────────────────
# Compute mean default rate per state from TRAIN ONLY.
# This is the key leakage-prevention step — never use test labels here.

if 'addr_state' in X_train.columns:
    state_default_rate = (
        pd.concat([X_train[['addr_state']], y_train], axis=1)
        .groupby('addr_state')['target']
        .mean()
        .rename('addr_state_default_rate')
    )

    global_mean = y_train.mean()  # fallback for unseen states in test

    X_train['addr_state'] = X_train['addr_state'].map(state_default_rate)
    X_test['addr_state']  = X_test['addr_state'].map(state_default_rate).fillna(global_mean)

    # Rename for clarity
    X_train.rename(columns={'addr_state': 'addr_state_default_rate'}, inplace=True)
    X_test.rename(columns={'addr_state': 'addr_state_default_rate'}, inplace=True)

    print('Top 10 states by default rate (train):')
    print(state_default_rate.sort_values(ascending=False).head(10).round(4))
else:
    print('addr_state not found — may have been dropped earlier')

Top 10 states by default rate (train):
addr_state
MS   0.2062
NE   0.2058
AR   0.2055
AL   0.2034
OK   0.2026
NV   0.1995
LA   0.1986
SD   0.1894
NM   0.1871
FL   0.1856
Name: addr_state_default_rate, dtype: float64


In [21]:
# earliest_cr_line is a date — parse to credit age in months, then drop original
# This belongs in Stage 4 properly, but we must prevent it entering OHE now
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
# For now: drop it entirely — Stage 4 will engineer credit_age_months from scratch
# We cannot compute credit_age_months here because we don't have issue_d anymore
X_train.drop(columns=['earliest_cr_line'], inplace=True)
X_test.drop(columns=['earliest_cr_line'], inplace=True)

In [22]:
# ── One-hot encode remaining low-cardinality categoricals ────────────────────
# pd.get_dummies on train, then reindex test to match train columns.
# This handles the case where test is missing a rare category — fills with 0.

ohe_cols = [
    c for c in X_train.select_dtypes(include='object').columns
    if c not in ['id','addr_state']  # already handled above
]

print(f'One-hot encoding {len(ohe_cols)} columns: {ohe_cols}')

# Check cardinality before encoding
for col in ohe_cols:
    print(f'  {col}: {X_train[col].nunique()} unique values')

One-hot encoding 7 columns: ['home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'initial_list_status', 'application_type', 'disbursement_method']
  home_ownership: 6 unique values
  verification_status: 3 unique values
  pymnt_plan: 2 unique values
  purpose: 14 unique values
  initial_list_status: 2 unique values
  application_type: 2 unique values
  disbursement_method: 2 unique values


In [23]:
# ── Apply one-hot encoding ────────────────────────────────────────────────────
X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True, dtype='int8')

# Align test to train columns — fills missing categories with 0
X_test  = pd.get_dummies(X_test, columns=ohe_cols, drop_first=True, dtype='int8')
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f'Shape after encoding — Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Column count match: {list(X_train.columns) == list(X_test.columns)}')

Shape after encoding — Train: (1057477, 80)  |  Test: (264370, 80)
Column count match: True


In [24]:
# Catch any float64 columns that survived the first downcast pass
float64_remaining = X_train.select_dtypes('float64').columns.tolist()
if float64_remaining:
    print(f'Fixing remaining float64: {float64_remaining}')
    X_train[float64_remaining] = X_train[float64_remaining].astype('float32')
    X_test[float64_remaining]  = X_test[float64_remaining].astype('float32')

Fixing remaining float64: ['annual_inc', 'addr_state_default_rate']


### Interpretation
Train and test must have **identical column lists** after this step — the `reindex` enforces this.  
If `Column count match: False`, there's a column order mismatch (not a missing column) — inspect with `set(X_train.columns).symmetric_difference(set(X_test.columns))`.

The `addr_state_default_rate` column should show meaningful variation — states like Mississippi and Nevada typically have higher default rates than Midwestern states. If all values are identical, the merge failed silently.

---
## 10. Final Sanity Checks

Before saving, run a final health check across four dimensions:  
shapes, dtypes, null counts, and target alignment.  
Any failure here means a bug upstream — better to catch it now than to debug from Stage 5.

In [25]:
# ── Final health check ────────────────────────────────────────────────────────
print('=' * 55)
print('FINAL SANITY CHECK')
print('=' * 55)

# 1. Shapes
print(f'\n[1] Shapes')
print(f'    X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'    X_test:  {X_test.shape}   y_test:  {y_test.shape}')

# 2. Column alignment
print(f'\n[2] Column alignment')
mismatch = set(X_train.columns).symmetric_difference(set(X_test.columns))
print(f'    Mismatched columns: {mismatch if mismatch else "None ✓"}')

# 3. Null check
print(f'\n[3] Null check')
print(f'    Train nulls: {X_train.isna().sum().sum()}')
print(f'    Test nulls:  {X_test.isna().sum().sum()}')

# 4. Target distribution
print(f'\n[4] Target distribution')
print(f'    Train default rate: {y_train.mean():.4f}')
print(f'    Test default rate:  {y_test.mean():.4f}')

# 5. Memory
mem_train = X_train.memory_usage(deep=True).sum() / 1e6
mem_test  = X_test.memory_usage(deep=True).sum()  / 1e6
print(f'\n[5] Memory')
print(f'    X_train: {mem_train:.1f} MB')
print(f'    X_test:  {mem_test:.1f} MB')

# 6. Dtype summary
print(f'\n[6] Dtype summary')
print(X_train.dtypes.value_counts())

FINAL SANITY CHECK

[1] Shapes
    X_train: (1057477, 80)  y_train: (1057477,)
    X_test:  (264370, 80)   y_test:  (264370,)

[2] Column alignment
    Mismatched columns: None ✓

[3] Null check
    Train nulls: 0
    Test nulls:  0

[4] Target distribution
    Train default rate: 0.1722
    Test default rate:  0.1722

[5] Memory
    X_train: 298.2 MB
    X_test:  74.6 MB

[6] Dtype summary
float32    53
int8       26
object      1
Name: count, dtype: int64


### Interpretation
**What to look for:**
- Train/test default rates should differ by <0.1% — stratification guarantee
- Zero mismatched columns — reindex alignment working
- Zero nulls — imputation complete
- No `float64` dtypes remaining — downcast was effective
- X_train memory target: <500 MB for this dataset size

If `float64` columns remain, run `X_train.select_dtypes('float64').columns` to identify them — likely newly-created OHE columns that defaulted to float.

---
## 11. Save Artifacts

We save four objects:
1. **`02_cleaned.parquet`** — the full cleaned dataframe (train + test rows, features + target). Downstream notebooks reload and re-split using the saved indices.
2. **`02_split_indices.json`** — already saved in Section 7.
3. **`02_target_encoding_map.json`** — the state→default_rate map, needed to encode new data at inference time.
4. **`02_imputation_values.json`** — medians and modes from train, needed at inference time.

Saving the encoding maps is a professional habit — without them, you can't score a new loan in production without re-running this entire notebook.

In [26]:
# ── Reassemble full cleaned dataframe for saving ──────────────────────────────
# Combine train and test back together; downstream notebooks re-split via indices.
df_out = pd.concat([
    X_train.assign(target=y_train.values),
    X_test.assign(target=y_test.values)
], axis=0)

# ── Save main parquet ─────────────────────────────────────────────────────────
out_path = DATA_DIR / '02_cleaned.parquet'
df_out.to_parquet(out_path, index=True)
print(f'Saved: {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {df_out.shape}')

Saved: data/02_cleaned.parquet  (99.8 MB)
Shape: (1321847, 81)


In [27]:
# ── Save target encoding map ──────────────────────────────────────────────────
if 'state_default_rate' in dir():
    te_map = {
        'state_default_rates': state_default_rate.to_dict(),
        'global_mean': float(global_mean)
    }
    with open(DATA_DIR / '02_target_encoding_map.json', 'w') as f:
        json.dump(te_map, f, indent=2)
    print('Saved: data/02_target_encoding_map.json')

# ── Save imputation values ────────────────────────────────────────────────────
imputation_values = {
    'medians': medians.to_dict() if 'medians' in dir() else {},
    'modes':   modes.to_dict()   if 'modes'   in dir() else {},
    'sentinel_cols': sentinel_cols,
    'sentinel_value': 999
}
with open(DATA_DIR / '02_imputation_values.json', 'w') as f:
    json.dump(imputation_values, f, indent=2, default=str)

print('Saved: data/02_imputation_values.json')
print('\n✅ All Stage 2 artifacts saved successfully.')

Saved: data/02_target_encoding_map.json
Saved: data/02_imputation_values.json

✅ All Stage 2 artifacts saved successfully.


---
## Stage 2 — Summary & Handoff to Stage 3

### What was done
| Step | Decision | Rationale |
|---|---|---|
| Column filter at load | Excluded leakage + >50% missing + identifiers | Never load what you don't need |
| Year filter | Kept issue_year ≤ 2016 only | Avoid label maturity problem |
| Target construction | Mapped loan_status → binary per Stage 0 definition | Ground truth alignment |
| Stratified split | 80/20, stratify on target, seed=42 | Reproducible, balanced evaluation |
| Sentinel imputation | mths_since_* → 999 | No-history is a meaningful state |
| Median imputation | Continuous numerics | Robust to income/balance outliers |
| Target encoding | addr_state → mean default rate (train only) | 1 column, real signal, no leakage |
| One-hot encoding | Low-cardinality nominals | No false ordinality imposed |

### Artifacts for Stage 3
- Load `data/02_cleaned.parquet`
- Load `data/02_split_indices.json` to reconstruct the same train/test split
- Feature selection will operate on train set only — all selection statistics fit on train, evaluated on test

### Open questions for Stage 3
- `funded_amnt` vs `loan_amnt`: expected near-perfect correlation — confirm and drop one
- `int_rate` vs `sub_grade`: int_rate had the highest effect size (d=0.62) but encodes LC's own risk model — assess whether it should be kept, removed, or both retained
- `inq_last_6mths`: ranked 3rd in effect size but no dedicated analysis yet — examine bucketed default rates in Stage 3 before deciding